In [1]:

print("Spark session connected!")

print("Spark Version:", spark.version)

print("Application ID:", spark.sparkContext.applicationId)


StatementMeta(olistsparkjob, 9, 2, Finished, Available, Finished, False)

Spark session connected!
Spark Version: 3.5.5.5.4.20260824.232707473
Application ID: application_1795191036577_0002


In [2]:

from pyspark.sql import functions as F

silver_path = "abfss://silver@olistdatalake2027.dfs.core.windows.net/"

# Load Silver datasets
orders = spark.read.parquet(silver_path + "orders/")
order_items = spark.read.parquet(silver_path + "order_items/")
customers = spark.read.parquet(silver_path + "customers/")
products = spark.read.parquet(silver_path + "products/")
payments = spark.read.parquet(silver_path + "payments/")

print("All Silver datasets loaded successfully!")

print("Orders:", orders.count())
print("Order Items:", order_items.count())
print("Customers:", customers.count())
print("Products:", products.count())
print("Payments:", payments.count())


StatementMeta(olistsparkjob, 9, 3, Finished, Available, Finished, False)

All Silver datasets loaded successfully!
Orders: 99441
Order Items: 112650
Customers: 99441
Products: 32951
Payments: 103884


In [3]:

# Convert columns to appropriate data types
order_items = order_items.withColumn(
    "price", F.col("price").cast("double")
).withColumn(
    "freight_value", F.col("freight_value").cast("double")
)

orders = orders.withColumn(
    "order_purchase_timestamp",
    F.to_timestamp("order_purchase_timestamp")
)

# Calculate revenue for each order
order_revenue = order_items.groupBy("order_id").agg(
    F.sum("price").alias("product_revenue"),
    F.sum("freight_value").alias("freight_revenue")
)

# Join with orders
revenue_df = orders.join(
    order_revenue,
    on="order_id",
    how="inner"
)

# Calculate total revenue
revenue_df = revenue_df.withColumn(
    "total_revenue",
    F.col("product_revenue") + F.col("freight_revenue")
)

# Extract month
revenue_df = revenue_df.withColumn(
    "revenue_month",
    F.date_format("order_purchase_timestamp", "yyyy-MM")
)

print("Revenue Analysis Completed")

display(revenue_df.limit(10))


StatementMeta(olistsparkjob, 9, 4, Finished, Available, Finished, False)

Revenue Analysis Completed


SynapseWidget(Synapse.DataFrame, 0bdb0453-763e-4ff9-ac79-f17686b6da53)

In [4]:

monthly_revenue = revenue_df.groupBy("revenue_month").agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.round(F.sum("product_revenue"), 2).alias("product_revenue"),
    F.round(F.sum("freight_revenue"), 2).alias("freight_revenue"),
    F.round(F.sum("total_revenue"), 2).alias("total_revenue")
).orderBy("revenue_month")

display(monthly_revenue)


StatementMeta(olistsparkjob, 9, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9093bf47-904d-4d71-bb0d-b3d9975aba55)

In [5]:

gold_path = "abfss://gold@olistdatalake2027.dfs.core.windows.net/"

monthly_revenue.write \
    .mode("overwrite") \
    .parquet(gold_path + "revenue_analysis/")

print("Gold Revenue Analysis Saved Successfully!")


StatementMeta(olistsparkjob, 9, 6, Finished, Available, Finished, False)

Gold Revenue Analysis Saved Successfully!


In [6]:

revenue_gold = spark.read.parquet(
    gold_path + "revenue_analysis/"
)

print("Gold Revenue Records:", revenue_gold.count())

display(revenue_gold)


StatementMeta(olistsparkjob, 9, 7, Finished, Available, Finished, False)

Gold Revenue Records: 24


SynapseWidget(Synapse.DataFrame, 36dcf580-1094-442e-b114-e20dd185fb8d)

In [7]:

from pyspark.sql import functions as F

silver_path = "abfss://silver@olistdatalake2027.dfs.core.windows.net/"

customers = spark.read.parquet(silver_path + "customers/")
orders = spark.read.parquet(silver_path + "orders/")
order_items = spark.read.parquet(silver_path + "order_items/")

print("Silver datasets loaded successfully!")


StatementMeta(olistsparkjob, 9, 8, Finished, Available, Finished, False)

Silver datasets loaded successfully!


In [8]:

customer_orders = orders.groupBy("customer_id").agg(
    F.countDistinct("order_id").alias("total_orders")
)

display(customer_orders.limit(10))


StatementMeta(olistsparkjob, 9, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f1696247-fb76-438b-8940-977efff7a52a)

In [9]:

order_revenue = order_items.groupBy("order_id").agg(
    F.sum("price").alias("product_revenue"),
    F.sum("freight_value").alias("freight_revenue")
)

order_revenue = order_revenue.withColumn(
    "total_revenue",
    F.col("product_revenue") + F.col("freight_revenue")
)

customer_revenue = orders.join(
    order_revenue,
    on="order_id",
    how="inner"
)

customer_revenue = customer_revenue.groupBy("customer_id").agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.round(F.sum("total_revenue"), 2).alias("total_spent")
)

display(customer_revenue.limit(10))


StatementMeta(olistsparkjob, 9, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b8b875a5-90ff-480d-a01e-86ef5ab2e1fc)

In [10]:

customer_gold = customers.join(
    customer_revenue,
    on="customer_id",
    how="left"
)

customer_gold = customer_gold.fillna({
    "total_orders": 0,
    "total_spent": 0.0
})

customer_gold = customer_gold.withColumn(
    "customer_segment",
    F.when(F.col("total_orders") >= 3, "Frequent Customer")
     .when(F.col("total_orders") == 2, "Repeat Customer")
     .otherwise("One-time Customer")
)

display(customer_gold.limit(10))


StatementMeta(olistsparkjob, 9, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 54959417-2600-4265-a4c8-7bff11c4fb49)

In [11]:

state_analysis = customer_gold.groupBy(
    "customer_state"
).agg(
    F.countDistinct("customer_unique_id").alias("unique_customers"),
    F.round(F.sum("total_spent"), 2).alias("total_revenue")
).orderBy(
    F.desc("total_revenue")
)

display(state_analysis)


StatementMeta(olistsparkjob, 9, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1ddeb988-5de8-4593-a94f-7120d34849ad)

In [12]:

gold_path = "abfss://gold@olistdatalake2027.dfs.core.windows.net/"

customer_gold.write \
    .mode("overwrite") \
    .parquet(gold_path + "customer_analysis/")

state_analysis.write \
    .mode("overwrite") \
    .parquet(gold_path + "state_customer_analysis/")

print("Customer Analysis Gold Saved Successfully!")


StatementMeta(olistsparkjob, 9, 13, Finished, Available, Finished, False)

Customer Analysis Gold Saved Successfully!


In [13]:

customer_check = spark.read.parquet(
    gold_path + "customer_analysis/"
)

print("Customer Gold Records:", customer_check.count())

display(customer_check.limit(10))


StatementMeta(olistsparkjob, 9, 14, Finished, Available, Finished, False)

Customer Gold Records: 99441


SynapseWidget(Synapse.DataFrame, 10b31685-28ae-471c-8435-7aa41f9c6c51)

In [14]:

from pyspark.sql import functions as F

silver_path = "abfss://silver@olistdatalake2027.dfs.core.windows.net/"

order_items = spark.read.parquet(silver_path + "order_items/")
products = spark.read.parquet(silver_path + "products/")

print("Silver datasets loaded successfully!")

print("Order Items:", order_items.count())
print("Products:", products.count())


StatementMeta(olistsparkjob, 9, 15, Finished, Available, Finished, False)

Silver datasets loaded successfully!
Order Items: 112650
Products: 32951


In [15]:

product_sales = order_items.join(
    products,
    on="product_id",
    how="left"
)

display(product_sales.limit(10))


StatementMeta(olistsparkjob, 9, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c1b657b9-1c08-4140-a1e3-565ba95ed395)

In [16]:

product_analysis = product_sales.groupBy(
    "product_id",
    "product_category_name"
).agg(
    F.count("*").alias("total_items_sold"),
    F.sum("price").alias("total_revenue"),
    F.sum("freight_value").alias("total_freight"),
    F.round(F.avg("price"), 2).alias("average_price")
)

product_analysis = product_analysis.withColumn(
    "total_revenue",
    F.round(F.col("total_revenue"), 2)
)

display(
    product_analysis.orderBy(
        F.desc("total_revenue")
    ).limit(20)
)


StatementMeta(olistsparkjob, 9, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b0651b8c-defa-4f00-a804-f5f385c898d9)

In [17]:

top_products = product_analysis.orderBy(
    F.desc("total_revenue")
).limit(10)

display(top_products)


StatementMeta(olistsparkjob, 9, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4de042a0-b033-4a74-98c2-d50aea71d6ef)

In [18]:

category_analysis = product_analysis.groupBy(
    "product_category_name"
).agg(
    F.sum("total_items_sold").alias("total_items_sold"),
    F.round(F.sum("total_revenue"), 2).alias("category_revenue")
).orderBy(
    F.desc("category_revenue")
)

display(category_analysis.limit(20))


StatementMeta(olistsparkjob, 9, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 621917bb-5db0-4d7a-b4f0-85722a38393b)

In [19]:

gold_path = "abfss://gold@olistdatalake2027.dfs.core.windows.net/"

product_analysis.write \
    .mode("overwrite") \
    .parquet(gold_path + "product_analysis/")

top_products.write \
    .mode("overwrite") \
    .parquet(gold_path + "top_products/")

category_analysis.write \
    .mode("overwrite") \
    .parquet(gold_path + "category_analysis/")

print("Product Performance Gold Saved Successfully!")


StatementMeta(olistsparkjob, 9, 20, Finished, Available, Finished, False)

Product Performance Gold Saved Successfully!


In [20]:

product_gold = spark.read.parquet(
    gold_path + "product_analysis/"
)

print("Product Gold Records:", product_gold.count())

display(product_gold.limit(10))


StatementMeta(olistsparkjob, 9, 21, Finished, Available, Finished, False)

Product Gold Records: 32951


SynapseWidget(Synapse.DataFrame, 6ebc0c2e-e899-44b5-b6ac-7ac64c9b7339)

In [21]:

product_gold = spark.read.parquet(
    gold_path + "product_analysis/"
)

print("Product Gold Records:", product_gold.count())

display(product_gold.limit(10))


StatementMeta(olistsparkjob, 9, 22, Finished, Available, Finished, False)

Product Gold Records: 32951


SynapseWidget(Synapse.DataFrame, 33132427-f804-4b60-9e4e-af1544fc7df0)

In [22]:

from pyspark.sql import functions as F

silver_path = "abfss://silver@olistdatalake2027.dfs.core.windows.net/"

payments = spark.read.parquet(silver_path + "payments/")
orders = spark.read.parquet(silver_path + "orders/")

print("Payment datasets loaded successfully!")

print("Payments:", payments.count())
print("Orders:", orders.count())


StatementMeta(olistsparkjob, 9, 23, Finished, Available, Finished, False)

Payment datasets loaded successfully!
Payments: 103884
Orders: 99441


In [23]:

payment_method_analysis = payments.groupBy(
    "payment_type"
).agg(
    F.count("*").alias("total_transactions"),
    F.countDistinct("order_id").alias("total_orders"),
    F.round(F.sum("payment_value"), 2).alias("total_payment"),
    F.round(F.avg("payment_value"), 2).alias("average_payment")
).orderBy(
    F.desc("total_payment")
)

display(payment_method_analysis)


StatementMeta(olistsparkjob, 9, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a1eada45-0267-4bb7-b5ad-772b7aea95d2)

In [24]:

installment_analysis = payments.groupBy(
    "payment_installments"
).agg(
    F.count("*").alias("total_transactions"),
    F.round(F.sum("payment_value"), 2).alias("total_payment"),
    F.round(F.avg("payment_value"), 2).alias("average_payment")
).orderBy(
    "payment_installments"
)

display(installment_analysis)


StatementMeta(olistsparkjob, 9, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dd352253-642f-4f44-9595-e94214ce6df5)

In [25]:

payment_orders = payments.join(
    orders.select("order_id", "order_purchase_timestamp"),
    on="order_id",
    how="inner"
)

payment_orders = payment_orders.withColumn(
    "payment_month",
    F.date_format(
        F.to_timestamp("order_purchase_timestamp"),
        "yyyy-MM"
    )
)

monthly_payment = payment_orders.groupBy(
    "payment_month"
).agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.round(F.sum("payment_value"), 2).alias("monthly_payment")
).orderBy("payment_month")

display(monthly_payment)


StatementMeta(olistsparkjob, 9, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9047323a-4ca0-4933-a035-168c6c178c1c)

In [26]:

gold_path = "abfss://gold@olistdatalake2027.dfs.core.windows.net/"

payment_method_analysis.write \
    .mode("overwrite") \
    .parquet(gold_path + "payment_method_analysis/")

installment_analysis.write \
    .mode("overwrite") \
    .parquet(gold_path + "installment_analysis/")

monthly_payment.write \
    .mode("overwrite") \
    .parquet(gold_path + "monthly_payment_analysis/")

print("Payment Analysis Gold Saved Successfully!")


StatementMeta(olistsparkjob, 9, 27, Finished, Available, Finished, False)

Payment Analysis Gold Saved Successfully!


In [27]:

payment_gold = spark.read.parquet(
    gold_path + "payment_method_analysis/"
)

print("Payment Gold Records:", payment_gold.count())

display(payment_gold)


StatementMeta(olistsparkjob, 9, 28, Finished, Available, Finished, False)

Payment Gold Records: 5


SynapseWidget(Synapse.DataFrame, a9084473-9c9f-407e-a9d0-60bb8a07d2f5)

In [28]:

from pyspark.sql import functions as F

silver_path = "abfss://silver@olistdatalake2027.dfs.core.windows.net/"

orders = spark.read.parquet(silver_path + "orders/")

print("Orders loaded successfully!")
print("Total records:", orders.count())

display(orders.limit(5))


StatementMeta(olistsparkjob, 9, 29, Finished, Available, Finished, False)

Orders loaded successfully!
Total records: 99441


SynapseWidget(Synapse.DataFrame, 8431cb9e-917f-4370-850f-4d29426cee1a)

In [29]:

date_columns = [
    "order_purchase_timestamp",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col_name in date_columns:
    orders = orders.withColumn(
        col_name,
        F.to_timestamp(F.col(col_name))
    )

print("Date conversion completed!")


StatementMeta(olistsparkjob, 9, 30, Finished, Available, Finished, False)

Date conversion completed!


In [30]:

delivery_df = orders.withColumn(
    "delivery_days",
    F.datediff(
        F.to_date("order_delivered_customer_date"),
        F.to_date("order_purchase_timestamp")
    )
)

delivery_df = delivery_df.withColumn(
    "estimated_delivery_days",
    F.datediff(
        F.to_date("order_estimated_delivery_date"),
        F.to_date("order_purchase_timestamp")
    )
)

display(delivery_df.limit(10))


StatementMeta(olistsparkjob, 9, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8aa5bfe0-7ea8-4020-806e-10e1a234b84a)

In [31]:

delivery_df = delivery_df.withColumn(
    "delivery_status",
    F.when(
        F.col("order_delivered_customer_date").isNull(),
        "Not Delivered"
    ).when(
        F.to_date("order_delivered_customer_date") >
        F.to_date("order_estimated_delivery_date"),
        "Late Delivery"
    ).otherwise("On-Time Delivery")
)

display(delivery_df.select(
    "order_id",
    "delivery_days",
    "estimated_delivery_days",
    "delivery_status"
).limit(20))


StatementMeta(olistsparkjob, 9, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 30cbf084-9e81-445e-ba4e-828d1e1db241)

In [32]:

delivery_summary = delivery_df.groupBy(
    "delivery_status"
).agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.round(F.avg("delivery_days"), 2).alias("average_delivery_days")
)

display(delivery_summary)


StatementMeta(olistsparkjob, 9, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 38542b5d-28ab-4ac3-a76d-0975e7d5a727)

In [33]:

monthly_delivery = delivery_df.groupBy(
    F.date_format(
        "order_purchase_timestamp", "yyyy-MM"
    ).alias("order_month")
).agg(
    F.countDistinct("order_id").alias("total_orders"),
    F.round(F.avg("delivery_days"), 2).alias("avg_delivery_days"),
    F.sum(
        F.when(F.col("delivery_status") == "Late Delivery", 1)
         .otherwise(0)
    ).alias("late_orders")
).orderBy("order_month")

display(monthly_delivery)


StatementMeta(olistsparkjob, 9, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e1df5174-17d1-46e1-8964-7f5ac9262e5a)

In [34]:

gold_path = "abfss://gold@olistdatalake2027.dfs.core.windows.net/"

delivery_df.write \
    .mode("overwrite") \
    .parquet(gold_path + "delivery_analysis/")

delivery_summary.write \
    .mode("overwrite") \
    .parquet(gold_path + "delivery_summary/")

monthly_delivery.write \
    .mode("overwrite") \
    .parquet(gold_path + "monthly_delivery_analysis/")

print("Delivery Performance Gold Saved Successfully!")


StatementMeta(olistsparkjob, 9, 35, Finished, Available, Finished, False)

Delivery Performance Gold Saved Successfully!


In [35]:

delivery_gold = spark.read.parquet(
    gold_path + "delivery_analysis/"
)

print("Delivery Gold Records:", delivery_gold.count())

display(delivery_gold.limit(10))


StatementMeta(olistsparkjob, 9, 36, Finished, Available, Finished, False)

Delivery Gold Records: 99441


SynapseWidget(Synapse.DataFrame, 36abf42f-f91b-4a3b-9bda-980a873dcfbb)

In [1]:
from pyspark.sql.functions import *

orders = spark.read.parquet(
    "abfss://silver@olistdatalake2027.dfs.core.windows.net/orders/"
)

display(orders.limit(5))

StatementMeta(olistsparkjob, 10, 2, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cbda4d09-d511-42e1-be15-8330c5f9ec85)

In [2]:
delivery = orders.withColumn(
    "delivery_days",
    datediff(
        col("order_delivered_customer_date"),
        col("order_purchase_timestamp")
    )
).withColumn(
    "estimated_delivery_days",
    datediff(
        col("order_estimated_delivery_date"),
        col("order_purchase_timestamp")
    )
).withColumn(
    "delivery_delay_days",
    datediff(
        col("order_delivered_customer_date"),
        col("order_estimated_delivery_date")
    )
).withColumn(
    "delivery_status",
    when(col("delivery_delay_days") <= 0, "On Time")
    .otherwise("Delayed")
)

display(delivery.limit(10))

StatementMeta(olistsparkjob, 10, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 033a65b6-1958-4ea5-bdc3-6f768547694c)

In [3]:
delivery_kpis = delivery.agg(
    count("order_id").alias("total_orders"),
    round(avg("delivery_days"), 2).alias("avg_delivery_days"),
    sum(when(col("delivery_status") == "On Time", 1)
        .otherwise(0)).alias("on_time_orders"),
    sum(when(col("delivery_status") == "Delayed", 1)
        .otherwise(0)).alias("delayed_orders")
)

display(delivery_kpis)

StatementMeta(olistsparkjob, 10, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f2434258-1a51-4f2e-a8a8-d95dd639568a)

In [4]:
delivery.write \
    .mode("overwrite") \
    .parquet(
        "abfss://gold@olistdatalake2027.dfs.core.windows.net/delivery_performance/"
    )

StatementMeta(olistsparkjob, 10, 5, Finished, Available, Finished, False)